# 🔍 Deepfake Detection System — With Model Pickle Save/Load

**New in this version:** The trained model is saved as a `.pkl` (pickle) file AND as `.h5`.  
On next run, it **loads from pickle** — skipping training entirely.

### ⚡ Workflow
- **First run:** Trains the model, saves it as `deepfake_model.pkl`
- **Every later run:** Loads instantly from pickle — no retraining!

---

## ✅ Step 1: Install & Import Libraries

In [ ]:
!pip install opencv-python-headless -q
print('✅ Libraries ready!')

In [ ]:
import os, random, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
np.random.seed(42); tf.random.set_seed(42); random.seed(42)

# ── Paths ──
BASE_DIR    = '/content/deepfake_dataset'
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')
VAL_DIR     = os.path.join(BASE_DIR, 'val')
PICKLE_PATH = '/content/deepfake_model.pkl'     # Pickle file path
H5_PATH     = '/content/deepfake_model.h5'      # Keras H5 path (needed for pickle wrapper)

IMG_SIZE   = 224
BATCH_SIZE = 16
EPOCHS     = 5
NUM_TRAIN  = 150
NUM_VAL    = 50

print(f'✅ TensorFlow {tf.__version__} | GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

## ✅ Step 2: Generate Synthetic Dataset

In [ ]:
# Only generate dataset if it doesn't already exist
if not os.path.exists(os.path.join(TRAIN_DIR, 'real', 'real_0000.jpg')):
    for split in ['train', 'val']:
        for cls in ['real', 'fake']:
            os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)

    def make_real_image(path):
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img[:, :] = [random.randint(180,230), random.randint(140,180), random.randint(100,140)]
        noise = np.random.normal(0, 12, (IMG_SIZE, IMG_SIZE, 3)).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        cv2.circle(img, (random.randint(60,80), random.randint(80,100)), 15, (40,30,20), -1)
        cv2.circle(img, (random.randint(140,160), random.randint(80,100)), 15, (40,30,20), -1)
        img = cv2.GaussianBlur(img, (5,5), 0)
        cv2.imwrite(path, img)

    def make_fake_image(path):
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img[:IMG_SIZE//2, :] = [random.randint(100,200), random.randint(50,150), random.randint(50,150)]
        img[IMG_SIZE//2:, :] = [random.randint(50,150), random.randint(100,200), random.randint(100,200)]
        noise = np.random.randint(0, 80, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img = cv2.add(img, noise)
        cv2.line(img, (0, IMG_SIZE//2 + random.randint(-10,10)), (IMG_SIZE, IMG_SIZE//2), (0,255,0), 2)
        cv2.imwrite(path, img)

    print('🖼️  Generating dataset...')
    for i in range(NUM_TRAIN):
        make_real_image(os.path.join(TRAIN_DIR, 'real', f'real_{i:04d}.jpg'))
        make_fake_image(os.path.join(TRAIN_DIR, 'fake', f'fake_{i:04d}.jpg'))
    for i in range(NUM_VAL):
        make_real_image(os.path.join(VAL_DIR, 'real', f'real_{i:04d}.jpg'))
        make_fake_image(os.path.join(VAL_DIR, 'fake', f'fake_{i:04d}.jpg'))
    print('✅ Dataset created!')
else:
    print('✅ Dataset already exists — skipping generation.')

## ✅ Step 3: Data Generators

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255, horizontal_flip=True,
    rotation_range=10, zoom_range=0.1
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=True
)
val_gen = val_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
print(f'✅ Class mapping: {train_gen.class_indices}  (0=real, 1=fake)')

## ✅ Step 4: Smart Train OR Load from Pickle

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  PICKLE SAVE / LOAD LOGIC
#
#  WHY PICKLE?
#  Keras/TF models cannot be directly pickled because they contain C++ ops.
#  Solution: Save the model weights as H5, and store the H5 PATH in pickle.
#  The pickle file acts as a smart pointer — it records:
#    - The model config (architecture JSON)
#    - The H5 weights path
#    - Training metadata (accuracy, loss, epochs)
#  On reload: we load the H5 model and restore it instantly.
# ═══════════════════════════════════════════════════════════════════════════

def build_model():
    """Build and compile MobileNetV2 deepfake detector."""
    base = MobileNetV2(input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, weights='imagenet')
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid')(x)
    m = Model(inputs=base.input, outputs=out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m


def save_model_pickle(model, history, pickle_path, h5_path):
    """
    Save model to H5 file and record metadata in a pickle file.
    The pickle stores: h5 path + model config + training stats.
    """
    model.save(h5_path)   # Save actual weights
    metadata = {
        'h5_path'        : h5_path,
        'model_config'   : model.to_json(),
        'final_val_acc'  : float(max(history.history['val_accuracy'])),
        'final_val_loss' : float(min(history.history['val_loss'])),
        'epochs_trained' : len(history.history['accuracy']),
        'img_size'       : IMG_SIZE,
        'class_indices'  : train_gen.class_indices,
    }
    with open(pickle_path, 'wb') as f:
        pickle.dump(metadata, f)
    print(f'✅ Pickle saved  → {pickle_path}')
    print(f'✅ H5 weights    → {h5_path}')
    print(f'   Val accuracy  : {metadata["final_val_acc"]:.4f}')


def load_model_pickle(pickle_path):
    """
    Load model from pickle metadata + H5 weights.
    Returns the loaded Keras model and metadata dict.
    """
    with open(pickle_path, 'rb') as f:
        metadata = pickle.load(f)
    loaded = tf.keras.models.load_model(metadata['h5_path'])
    print(f'✅ Model loaded from pickle!')
    print(f'   H5 source      : {metadata["h5_path"]}')
    print(f'   Val accuracy   : {metadata["final_val_acc"]:.4f}')
    print(f'   Epochs trained : {metadata["epochs_trained"]}')
    return loaded, metadata


# ── MAIN LOGIC: Load if pickle exists, else train and save ──
if os.path.exists(PICKLE_PATH) and os.path.exists(H5_PATH):
    print('⚡ Pickle file found! Loading pre-trained model (no retraining needed)...')
    model, pkl_meta = load_model_pickle(PICKLE_PATH)
    history = None   # No history when loaded from pickle
else:
    print('🚀 No pickle found. Training from scratch...')
    model = build_model()
    early_stop = EarlyStopping(monitor='val_loss', patience=2,
                               restore_best_weights=True, verbose=1)
    history = model.fit(
        train_gen, epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[early_stop], verbose=1
    )
    print('\n✅ Training complete! Saving model...')
    save_model_pickle(model, history, PICKLE_PATH, H5_PATH)

print('\n✅ Model is ready for predictions!')

## ✅ Step 5: Plot Training Curves (if just trained)

In [ ]:
if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Training History', fontsize=16, fontweight='bold')
    axes[0].plot(history.history['accuracy'],     label='Train', color='royalblue', marker='o')
    axes[0].plot(history.history['val_accuracy'], label='Val',   color='darkorange', marker='s')
    axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['loss'],         label='Train', color='royalblue', marker='o')
    axes[1].plot(history.history['val_loss'],     label='Val',   color='darkorange', marker='s')
    axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('ℹ️  Model loaded from pickle — no training history to plot.')
    print('   Training metrics from saved session:')
    print(f'   Val Accuracy : {pkl_meta["final_val_acc"]:.4f}')
    print(f'   Val Loss     : {pkl_meta["final_val_loss"]:.4f}')

## ✅ Step 6: Prediction Functions

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv'}

def preprocess_frame(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
    return np.expand_dims(resized.astype(np.float32) / 255.0, axis=0)


def predict_image(image_path):
    """Predict REAL or FAKE for a single image file."""
    try:
        if not os.path.exists(image_path):
            print(f'❌ File not found: {image_path}'); return None
        ext = os.path.splitext(image_path)[1].lower()
        if ext not in IMAGE_EXTS:
            print(f'❌ Unsupported format: {ext}'); return None
        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            print(f'❌ Cannot read image: {image_path}'); return None

        raw  = float(model.predict(preprocess_frame(img_bgr), verbose=0)[0][0])
        label = 'FAKE' if raw >= 0.5 else 'REAL'
        conf  = raw if label == 'FAKE' else 1.0 - raw

        fig, ax = plt.subplots(figsize=(4,4))
        ax.imshow(cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),(300,300)))
        ax.set_title(f'Prediction: {label}\nConfidence: {conf:.2%}',
                     color='red' if label=='FAKE' else 'green',
                     fontsize=13, fontweight='bold')
        ax.axis('off'); plt.tight_layout(); plt.show()

        print(f'\n  Prediction : {"🔴 FAKE" if label=="FAKE" else "🟢 REAL"}')
        print(f'  Confidence : {conf:.4f} ({conf:.2%})')
        return {'label': label, 'confidence': conf, 'raw_score': raw}
    except Exception as e:
        print(f'❌ Error in predict_image: {e}'); return None


def predict_video(video_path, max_frames=30, frame_skip=10):
    """Predict REAL or FAKE for a video using majority voting."""
    try:
        if not os.path.exists(video_path):
            print(f'❌ File not found: {video_path}'); return None
        ext = os.path.splitext(video_path)[1].lower()
        if ext not in VIDEO_EXTS:
            print(f'❌ Unsupported format: {ext}'); return None

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f'❌ Cannot open video: {video_path}'); return None

        preds, scores, frames_vis = [], [], []
        fc = 0
        while cap.isOpened() and len(preds) < max_frames:
            ret, frame = cap.read()
            if not ret: break
            fc += 1
            if fc % frame_skip != 0: continue
            raw   = float(model.predict(preprocess_frame(frame), verbose=0)[0][0])
            label = 'FAKE' if raw >= 0.5 else 'REAL'
            preds.append(label); scores.append(raw)
            if len(frames_vis) < 6:
                frames_vis.append((cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), label, raw))
        cap.release()

        if not preds:
            print('❌ No frames extracted.'); return None

        fake_n = preds.count('FAKE')
        real_n = preds.count('REAL')
        avg    = float(np.mean(scores))
        final  = 'FAKE' if fake_n > real_n else 'REAL'
        conf   = avg if final == 'FAKE' else 1.0 - avg

        # Visualize sample frames
        if frames_vis:
            nc = min(3, len(frames_vis))
            nr = (len(frames_vis)+nc-1)//nc
            fig, axes = plt.subplots(nr, nc, figsize=(4*nc, 4*nr))
            axes = np.array(axes).flatten()
            for i,(frm,lbl,sc) in enumerate(frames_vis):
                axes[i].imshow(cv2.resize(frm,(200,200)))
                c = sc if lbl=='FAKE' else 1.0-sc
                axes[i].set_title(f'{lbl}\n{c:.2%}',
                    color='red' if lbl=='FAKE' else 'green', fontweight='bold')
                axes[i].axis('off')
            for i in range(len(frames_vis), len(axes)): axes[i].axis('off')
            plt.suptitle('Video Frame Analysis', fontsize=14, fontweight='bold')
            plt.tight_layout(); plt.show()

        print(f'\n  Frames Analyzed : {len(preds)}')
        print(f'  FAKE frames     : {fake_n} | REAL frames: {real_n}')
        print(f'  Final Decision  : {"🔴 FAKE" if final=="FAKE" else "🟢 REAL"}')
        print(f'  Confidence      : {conf:.4f} ({conf:.2%})')
        return {'label': final, 'confidence': conf, 'fake_frames': fake_n,
                'real_frames': real_n, 'total_frames': len(preds)}
    except Exception as e:
        print(f'❌ Error in predict_video: {e}'); return None


print('✅ Prediction functions ready!')

## ✅ Step 7: Quick Self-Test on Dataset Samples

In [ ]:
print('── Testing REAL image ──')
predict_image(os.path.join(VAL_DIR, 'real', 'real_0000.jpg'))
print('\n── Testing FAKE image ──')
predict_image(os.path.join(VAL_DIR, 'fake', 'fake_0000.jpg'))

## ✅ Step 8: Upload Your Own Image or Video

In [ ]:
from google.colab import files as colab_files

print('═'*55)
print('  🔍 Upload any image (.jpg/.png) or video (.mp4/.avi)')
print('  System auto-detects type and runs correct predictor')
print('═'*55)

try:
    uploaded = colab_files.upload()
    if not uploaded:
        print('⚠️  No file uploaded.')
    else:
        for fname, content in uploaded.items():
            fpath = f'/content/{fname}'
            with open(fpath, 'wb') as f: f.write(content)
            ext = os.path.splitext(fname)[1].lower()
            print(f'\n📂 File: {fname} ({len(content)/1024:.1f} KB)')
            if ext in IMAGE_EXTS:
                print('   → Image detected → running predict_image()')
                predict_image(fpath)
            elif ext in VIDEO_EXTS:
                print('   → Video detected → running predict_video()')
                predict_video(fpath)
            else:
                print(f'❌ Unsupported format: {ext}')
except Exception as e:
    print(f'❌ Error: {e}')

## ✅ Step 9: Download Model Files (for Streamlit Deployment)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Download both files for use in Streamlit app
#  deepfake_model.pkl  → metadata pointer
#  deepfake_model.h5   → actual weights
#  Upload BOTH files to your GitHub repo / Streamlit deployment
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import files

print('📦 Downloading model files...')
print('   Save these to your Streamlit project folder!')
print()

if os.path.exists(H5_PATH):
    files.download(H5_PATH)
    print(f'✅ Downloaded: deepfake_model.h5  ({os.path.getsize(H5_PATH)/1024/1024:.1f} MB)')
else:
    print('❌ H5 file not found — run Step 4 first.')

if os.path.exists(PICKLE_PATH):
    files.download(PICKLE_PATH)
    print(f'✅ Downloaded: deepfake_model.pkl ({os.path.getsize(PICKLE_PATH)/1024:.1f} KB)')
else:
    print('❌ Pickle file not found — run Step 4 first.')

print('\n📌 Next step: Upload these files to your Streamlit GitHub repo.')

---
## 📋 Notes
- `deepfake_model.pkl` stores metadata (val accuracy, config, h5 path)
- `deepfake_model.h5` stores the actual Keras model weights
- Both files are needed for the Streamlit app to work
- Place both in the root of your GitHub repository